# Classificação binária de imagens: Pokémon de Fogo vs Água

**Pergunta:** dá para adivinhar se um Pokémon é do tipo Fogo ou Água só olhando a imagem?

**Modelos:** SVM, Random Forest, KNN e Gradient Boosting (ML clássico)
**Features:** cor (histograma HSV, cores dominantes, LAB) e forma (HOG), comparadas entre si
**Extras:** ajuste de parâmetros com validação aninhada, um teste de robustez com variantes shiny e em tons de cinza e uma extensão com a classe "Outro" (3 classes)
**Métricas:** acurácia, precisão, recall, F1, F2, ROC AUC e matriz de confusão
**Saída:** probabilidade de cada classe (`predict_proba`)

Classe positiva = **Fogo** (1). Classe negativa = **Água** (0).

> 🎤 **Resumo em 30 segundos:** "Peguei um dataset com imagens de Pokémon e seus tipos e treinei dois modelos clássicos, SVM e Random Forest, para dizer se o Pokémon é de Fogo ou de Água. Como os modelos não recebem imagem direto, extraí features de **cor** (histograma de matiz, saturação e brilho) e de **forma** (HOG). Comparei cor, forma e as duas juntas para ver o que mais pesa. [Complete aqui com o resultado da ablação depois de rodar: a hipótese é que a cor pese mais, já que Pokémon de fogo tendem a ser vermelhos e laranjas e os de água, azuis.]"

## 1. Setup

> 🎤 **O que falar:** instalo as bibliotecas e defino a semente aleatória (`SEED`), para o resultado ser reproduzível. Fixo a versão do scikit-learn para os modelos salvos funcionarem no app.

> ⚠️ Se a saída da célula de setup avisar que a versão não é a 1.9.1, vá em *Ambiente de execução → Reiniciar sessão* e depois *Executar tudo* (o Colab já pode ter carregado outra versão antes).

In [ ]:
# Fixa a versão do scikit-learn igual à do computador onde o app vai rodar: os modelos .joblib só carregam
# na MESMA versão. Para ver a do seu computador: python -c "import sklearn; print(sklearn.__version__)"
!pip install -q scikit-learn==1.9.1

In [ ]:
# kagglehub baixa o dataset direto do Kaggle; scikit-image fornece HOG e conversão de cores
!pip -q install kagglehub scikit-image

import os, re, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from PIL import Image
from sklearn.model_selection import StratifiedKFold, cross_val_predict   # validação cruzada estratificada
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler                          # padroniza as features (importante para SVM)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             fbeta_score, roc_auc_score, roc_curve,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42   # semente fixa: quem rodar de novo obtém o mesmo resultado

import sklearn
print('scikit-learn em uso:', sklearn.__version__)
if sklearn.__version__ != '1.9.1':
    print('ATENÇÃO: a versão não é a 1.9.1. Vá em Ambiente de execução > Reiniciar sessão e depois Executar tudo.')

## 2. Dataset
Baixa o dataset *Pokemon Images and Types* do Kaggle: uma imagem por Pokémon e um CSV com nome, tipo 1 e tipo 2.

**Como viramos um problema binário:** ficamos com os Pokémon que têm o tipo Fogo **ou** o tipo Água (no tipo 1 ou no tipo 2) e descartamos os que têm os dois ou nenhum.

> 🎤 **O que falar:** "O dataset tem centenas de Pokémon de vários tipos. Para ficar binário, filtrei só Fogo e Água. Um Pokémon como o Charizard é Fogo/Voador, então conta como Fogo. Quem tivesse os dois tipos ao mesmo tempo seria ambíguo, então fica de fora. Esse filtro deixa o conjunto **pequeno**, e isso vai influenciar como eu avalio."

In [ ]:
path = kagglehub.dataset_download('vishalsubbiah/pokemon-images-and-types')
print('Pasta do dataset:', path)

# Mostra a estrutura de pastas (ajuda a conferir e a depurar se algo estiver diferente do esperado)
for raiz, pastas, arquivos in os.walk(path):
    print(raiz, '->', len(arquivos), 'arquivos', arquivos[:3])

# Lê o CSV com os tipos (colunas esperadas: name, type1, type2)
csv_path = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)[0]
meta = pd.read_csv(csv_path)
meta.columns = [c.strip().lower() for c in meta.columns]
print(meta.head())

def norm(s):
    # Normaliza nomes (minúsculas, sem símbolos) para casar o nome do CSV com o nome do arquivo de imagem
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

imagens = {norm(os.path.splitext(os.path.basename(f))[0]): f
           for f in glob.glob(os.path.join(path, '**', '*.*'), recursive=True)
           if f.lower().endswith(('.png', '.jpg', '.jpeg'))}

meta['path'] = meta['name'].map(lambda n: imagens.get(norm(n)))
print('Pokémon sem imagem correspondente:', meta['path'].isna().sum())
meta = meta.dropna(subset=['path'])

In [ ]:
def tem_tipo(row, tipo):
    # True se o Pokémon tem esse tipo, seja no tipo 1 ou no tipo 2
    tipos = [str(row['type1']).strip().lower(), str(row['type2']).strip().lower()]
    return tipo in tipos

meta['fogo'] = meta.apply(lambda r: tem_tipo(r, 'fire'), axis=1)
meta['agua'] = meta.apply(lambda r: tem_tipo(r, 'water'), axis=1)

# XOR: fica quem tem Fogo OU Água, mas não os dois ao mesmo tempo
df = meta[meta.fogo ^ meta.agua].copy().reset_index(drop=True)
if len(df) == 0:
    raise ValueError('Nenhum Pokémon de Fogo/Água encontrado. Confira os nomes das colunas e dos tipos acima.')

df['label'] = df.fogo.astype(int)                           # 1 = Fogo, 0 = Água
df['classe'] = np.where(df.label == 1, 'Fogo', 'Água')

print(df.classe.value_counts())
print('Total:', len(df), 'imagens')

### Exemplos e balanceamento

> 🎤 **O que falar:** "Aqui estão exemplos de cada classe e a contagem. Há mais Pokémon de Água que de Fogo, então o conjunto é desbalanceado. Por isso uso `class_weight='balanced'` e não confio só na acurácia."

In [ ]:
# 5 exemplos de cada classe (linha de cima Água, linha de baixo Fogo)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for r, c in enumerate(['Água', 'Fogo']):
    amostras = df[df.classe == c].sample(5, random_state=SEED)
    for ax, (_, linha) in zip(axes[r], amostras.iterrows()):
        ax.imshow(Image.open(linha.path).convert('RGBA'))
        ax.set_title(f'{linha["name"]} ({c})', fontsize=9)
        ax.axis('off')
plt.tight_layout(); plt.show()

# Contagem por classe: evidencia o desbalanceamento
df.classe.value_counts().plot(kind='bar', title='Imagens por classe', color=['#4a90d9', '#e8603c'])
plt.show()

## 3. Extração de features (cor e forma)

SVM e Random Forest recebem uma **tabela de números**, não a imagem. Então transformamos cada imagem em um vetor com dois tipos de descrição:

- **Cor:** histograma de matiz (qual cor), saturação (quão viva) e brilho, calculado **só nos pixels do Pokémon** (o fundo transparente é ignorado), mais as cores médias.
- **Cor extra (`cor2`):** média e desvio no espaço de cor LAB, matiz médio (circular) e as **3 cores dominantes** da imagem (k-means).
- **Forma:** HOG (histograma de gradientes orientados), que descreve a direção das bordas.

Imagens sem transparência (fotos, JPG, sprites com fundo branco) têm o fundo branco ignorado, assim como o fundo transparente.

Comparamos cinco conjuntos: `cor`, `hog`, `cor+hog`, `cor+cor2` e `cor+cor2+hog`.

> 🎤 **O que falar:** "A hipótese é que cor importa mais que forma para o tipo. Por isso comparo vários conjuntos. Isso é uma **ablação**: tirar uma parte das features para ver quanto ela contribui."

O código das features fica num arquivo `features.py`, porque o app em Streamlit vai importar **exatamente o mesmo código**. Assim treino e app nunca ficam diferentes.

In [ ]:
%%writefile features.py
import warnings
import numpy as np
from PIL import Image
from skimage.feature import hog
from skimage.color import rgb2hsv, rgb2gray, rgb2lab
from sklearn.cluster import KMeans

SIZE = 128   # todas as imagens viram 128x128

def _abrir(img):
    # Aceita caminho de arquivo ou imagem PIL. Converte para RGBA (mantém a transparência)
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    return img.convert('RGBA').resize((SIZE, SIZE))

def _mascara(rgb, alpha):
    # Quais pixels são o Pokémon (e não o fundo)?
    if alpha.min() > 0.99:
        # Imagem opaca (JPG, sprite com fundo branco): ignora os pixels quase brancos
        mask = ~((rgb > 0.94).all(axis=-1))
    else:
        # Imagem com transparência: usa o canal alpha
        mask = alpha > 0.5
    if mask.sum() < 50:
        mask = np.ones(alpha.shape, dtype=bool)   # segurança: se a máscara ficar vazia, usa tudo
    return mask

def feats_cor(img):
    a = np.asarray(img, dtype=np.float32) / 255.0
    rgb, alpha = a[..., :3], a[..., 3]
    mask = _mascara(rgb, alpha)
    hsv = rgb2hsv(rgb)
    h, s, v = hsv[..., 0][mask], hsv[..., 1][mask], hsv[..., 2][mask]
    n = float(mask.sum())
    # Matiz só dos pixels com cor de verdade (ignora branco, cinza e preto, que não têm matiz útil)
    colorido = (s > 0.25) & (v > 0.2)
    hist_h = np.histogram(h[colorido], bins=18, range=(0, 1))[0] / n   # 18 faixas de matiz
    hist_s = np.histogram(s, bins=4, range=(0, 1))[0] / n              # 4 faixas de saturação
    hist_v = np.histogram(v, bins=4, range=(0, 1))[0] / n              # 4 faixas de brilho
    medias = np.concatenate([rgb[mask].mean(axis=0), [s.mean(), v.mean()]])
    return np.concatenate([hist_h, hist_s, hist_v, medias])           # 18 + 4 + 4 + 5 = 31 números

def feats_cor2(img):
    # Cores extras: LAB (percepção humana de cor), matiz médio circular e 3 cores dominantes
    a = np.asarray(img, dtype=np.float32) / 255.0
    rgb, alpha = a[..., :3], a[..., 3]
    mask = _mascara(rgb, alpha)
    px = rgb[mask]                                              # só os pixels do Pokémon, formato (n, 3)
    lab = rgb2lab(px.reshape(-1, 1, 3)).reshape(-1, 3)
    lab_stats = np.concatenate([lab.mean(axis=0), lab.std(axis=0)])            # 6 números
    hsv = rgb2hsv(px.reshape(-1, 1, 3)).reshape(-1, 3)
    h, s, v = hsv[:, 0], hsv[:, 1], hsv[:, 2]
    peso = s * (v > 0.2)                                        # matiz pesado pela saturação
    ang = 2 * np.pi * h                                         # matiz é circular: vermelho fica nas duas pontas
    sen = (peso * np.sin(ang)).sum() / (peso.sum() + 1e-6)
    cos = (peso * np.cos(ang)).sum() / (peso.sum() + 1e-6)
    circ = [sen, cos, np.hypot(sen, cos)]                       # 3 números (direção e força do matiz médio)
    # 3 cores dominantes por k-means (em no máximo 2000 pixels, para ser rápido e reproduzível)
    rng = np.random.RandomState(0)
    amostra = px if len(px) <= 2000 else px[rng.choice(len(px), 2000, replace=False)]
    if len(amostra) >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            km = KMeans(n_clusters=3, n_init=3, random_state=0).fit(amostra)
        cont = np.bincount(km.labels_, minlength=3).astype(float)
        ordem = np.argsort(-cont)                               # da cor mais frequente para a menos frequente
        dom = np.concatenate([[cont[i] / cont.sum()] + list(km.cluster_centers_[i]) for i in ordem])   # 3 x 4 = 12
    else:
        dom = np.zeros(12)
    return np.concatenate([lab_stats, circ, dom])               # 6 + 3 + 12 = 21 números

def feats_forma(img):
    a = np.asarray(img, dtype=np.float32) / 255.0
    rgb, alpha = a[..., :3], a[..., 3:4]
    sobre_branco = rgb * alpha + (1 - alpha)       # coloca o Pokémon sobre fundo branco
    gray = rgb2gray(sobre_branco)
    # HOG: direção das bordas em células de 32x32 pixels, normalizada em blocos de 2x2 células.
    # Células grandes geram poucas features (324), para a cor não ficar diluída
    return hog(gray, orientations=9, pixels_per_cell=(32, 32), cells_per_block=(2, 2))

def extrair(img, conjunto='cor+hog'):
    # conjunto: combinação de 'cor', 'cor2' e 'hog' separados por '+', ex.: 'cor+cor2+hog'
    nomes = conjunto.split('+')
    im = _abrir(img)
    partes = []
    if 'cor' in nomes:
        partes.append(feats_cor(im))
    if 'cor2' in nomes:
        partes.append(feats_cor2(im))
    if 'hog' in nomes:
        partes.append(feats_forma(im))
    return np.concatenate(partes)

In [ ]:
from features import extrair, SIZE

CONJUNTOS = ['cor', 'hog', 'cor+hog', 'cor+cor2', 'cor+cor2+hog']   # cor2 = cores extras (LAB, cores dominantes)
y = df.label.values                      # rótulos: 1 = Fogo, 0 = Água

# X[conjunto] = matriz (nº de imagens x nº de features)
X = {c: np.array([extrair(p, c) for p in df.path]) for c in CONJUNTOS}
for c in CONJUNTOS:
    print(c, X[c].shape)

## 4. Modelos e protocolo de avaliação

- **SVM (kernel RBF):** procura a fronteira que melhor separa as classes; a fronteira pode ser curva. Precisa de features padronizadas (`StandardScaler`).
- **Random Forest:** 300 árvores de decisão que votam; robusto e quase sem ajuste.
- **`class_weight='balanced'`:** dá mais peso à classe minoritária (Fogo).

**Validação cruzada estratificada (5 folds):** o conjunto é pequeno, então um teste de 15% teria só algumas dezenas de imagens e as métricas oscilariam muito. Na validação cruzada, os dados são divididos em 5 partes; cada parte é prevista por um modelo treinado nas outras 4. No fim, **toda imagem tem uma previsão feita por um modelo que nunca a viu**, e as métricas usam todas as imagens.

> 🎤 **O que falar:** "Com poucas imagens, separar um teste pequeno dá um resultado instável. A validação cruzada usa todos os dados para avaliar, sem nunca avaliar uma imagem com um modelo que a viu no treino. 'Estratificada' significa que cada parte mantém a proporção de Fogo e Água."

> ⚠️ **Limitação a mencionar:** Pokémon da mesma linha evolutiva (ex.: Charmander, Charmeleon, Charizard) são parecidos entre si e podem cair em partes diferentes, o que pode deixar o resultado um pouco otimista.

In [ ]:
def criar_modelos():
    # Cria modelos novos a cada chamada (evita reaproveitar um modelo já treinado por engano)
    return {
        # Pipeline: padroniza (média 0, desvio 1) e depois aplica o SVM.
        # C controla a rigidez da fronteira; gamma='scale' ajusta o alcance do kernel automaticamente.
        'SVM': make_pipeline(StandardScaler(),
                             SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced',
                                 probability=True, random_state=SEED)),
        'Random Forest': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                n_jobs=-1, random_state=SEED),
    }

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def metricas(y_real, proba, limiar):
    # Converte probabilidade em classe: proba >= limiar  ->  Fogo (1)
    pred = (proba >= limiar).astype(int)
    return {
        'Acurácia': accuracy_score(y_real, pred),
        'Precisão': precision_score(y_real, pred, zero_division=0),
        'Recall': recall_score(y_real, pred),
        'F1': f1_score(y_real, pred),
        'F2': fbeta_score(y_real, pred, beta=2),      # beta=2: recall pesa mais que a precisão
        'ROC AUC': roc_auc_score(y_real, proba),      # usa a probabilidade, não depende do limiar
    }

# Probabilidades "out-of-fold": para cada imagem, a probabilidade de Fogo prevista por um modelo que não a viu
oof = {}
for c in CONJUNTOS:
    for nome, m in criar_modelos().items():
        oof[(c, nome)] = cross_val_predict(m, X[c], y, cv=cv, method='predict_proba')[:, 1]
print('Pronto.')

### 4.1 Ablação: quais features ajudam? (limiar 0,5)

> 🎤 **O que falar:** "Esta tabela é o resultado central. Compare as linhas de `cor`, `hog` e `cor+hog` para cada modelo. Se `cor` chega perto de `cor+hog` e `hog` fica bem abaixo, a cor é o que carrega a informação do tipo. Depois compare `cor` com `cor+cor2`: se melhorar, as cores extras (LAB e cores dominantes) ajudam."

In [ ]:
linhas = [{'Features': c, 'Modelo': n, **metricas(y, p, 0.5)} for (c, n), p in oof.items()]
ablacao = pd.DataFrame(linhas).round(3)
ablacao

### 4.2 Métricas do conjunto principal (o melhor da ablação), com limiar 0,5 e limiar ajustado

**Dicionário das métricas** (classe positiva = Fogo):
- **Acurácia:** % de acertos no total. Engana com classes desbalanceadas.
- **Precisão:** dos que o modelo disse "Fogo", quantos eram.
- **Recall:** dos que eram Fogo, quantos o modelo achou.
- **F1:** média harmônica de precisão e recall.
- **F2:** como o F1, mas o recall pesa 2x mais.
- **ROC AUC:** capacidade de separar as classes em qualquer limiar (1,0 = perfeito, 0,5 = chute).
- **Limiar:** o modelo dá uma probabilidade; o limiar é o corte para decidir "Fogo". Baixar o limiar aumenta o recall e reduz a precisão.

> 🎤 **Sobre o F2:** "Pedi o F2 no trabalho. Aqui o problema é lúdico, então não há um custo real de errar, mas o F2 mostra o que acontece quando priorizo **não perder nenhum Pokémon de Fogo**: o recall sobe e a precisão cai. Escolhi o limiar que maximiza o F2."

> ⚠️ O conjunto de features e o limiar foram escolhidos nas mesmas previsões usadas para avaliar, o que é levemente otimista. Com conjunto pequeno é uma limitação aceitável, mas vale citar.

In [ ]:
# Conjunto principal = o de maior ROC AUC médio (entre SVM e RF) na ablação acima
PRINCIPAL = ablacao.groupby('Features')['ROC AUC'].mean().idxmax()
print('Conjunto principal escolhido pela ablação:', PRINCIPAL)
limiares = np.arange(0.05, 0.96, 0.01)     # candidatos a limiar: 0,05 até 0,95
thr_otimo, linhas = {}, []

for nome in ['SVM', 'Random Forest']:
    p = oof[(PRINCIPAL, nome)]
    f2s = [fbeta_score(y, (p >= t).astype(int), beta=2, zero_division=0) for t in limiares]
    thr_otimo[nome] = float(limiares[int(np.argmax(f2s))])      # limiar com maior F2
    linhas.append({'Modelo': nome, 'Limiar': 0.50, **metricas(y, p, 0.50)})
    linhas.append({'Modelo': nome, 'Limiar': round(thr_otimo[nome], 2), **metricas(y, p, thr_otimo[nome])})

resultados = pd.DataFrame(linhas).round(3)
resultados

### Matrizes de confusão (limiar ajustado para F2)

> 🎤 **Como ler a matriz de confusão:**
> - Linhas = classe **real**; colunas = classe **prevista**.
> - Canto superior esquerdo: Água classificado como Água (acerto). Canto inferior direito: Fogo classificado como Fogo (acerto).
> - Canto inferior esquerdo: **Fogo classificado como Água** (falso negativo, um Fogo que o modelo perdeu).
> - Canto superior direito: **Água classificado como Fogo** (falso positivo, alarme falso).
> - Compare os dois modelos: qual deixa escapar menos Pokémon de Fogo?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, nome in zip(axes, ['SVM', 'Random Forest']):
    pred = (oof[(PRINCIPAL, nome)] >= thr_otimo[nome]).astype(int)
    cm = confusion_matrix(y, pred)
    ConfusionMatrixDisplay(cm, display_labels=['Água', 'Fogo']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome} (limiar={thr_otimo[nome]:.2f})')
plt.tight_layout(); plt.show()

### Curva ROC

> 🎤 **Como ler:** a curva mostra, para todos os limiares, o recall (eixo y) contra a taxa de falsos positivos (eixo x). Quanto mais perto do canto superior esquerdo, melhor. A linha tracejada é o chute (AUC = 0,5).

In [ ]:
plt.figure(figsize=(6, 5))
for nome in ['SVM', 'Random Forest']:
    p = oof[(PRINCIPAL, nome)]
    fpr, tpr, _ = roc_curve(y, p)
    plt.plot(fpr, tpr, label=f'{nome} (AUC={roc_auc_score(y, p):.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray')       # referência: classificador aleatório
plt.xlabel('Taxa de falsos positivos'); plt.ylabel('Recall (taxa de verdadeiros positivos)')
plt.title('Curva ROC (validação cruzada)'); plt.legend(); plt.show()

### O que o modelo "olha"? Importância de cada faixa de matiz

> 🎤 **O que falar:** "O Random Forest permite ver quais features mais ajudaram. Cada barra é uma faixa de matiz (a cor da barra é a própria cor). Se as barras de vermelho e laranja forem altas, o modelo aprendeu que Pokémon de fogo são vermelhos e laranjas. Se as de azul forem altas, aprendeu que os de água são azuis."

In [ ]:
rf_cor = RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=SEED).fit(X['cor'], y)
imp = rf_cor.feature_importances_[:18]                    # as 18 primeiras features de cor são as faixas de matiz
cores = plt.cm.hsv(np.linspace(0, 1, 19)[:-1] + 1/36)     # cor de cada faixa (colormap hsv = o próprio matiz)
plt.figure(figsize=(9, 3.5))
plt.bar(range(18), imp, color=cores, edgecolor='gray')
plt.xticks([]); plt.ylabel('Importância'); plt.xlabel('Faixas de matiz (vermelho → laranja → verde → azul → roxo → vermelho)')
plt.title('Importância das faixas de cor (Random Forest)'); plt.show()

### Onde o modelo erra?

Mostra os Pokémon que o modelo classificou errado, com a probabilidade de Fogo que ele deu. Os erros mais "confiantes" aparecem primeiro.

> 🎤 **O que falar:** "Entender os erros é tão importante quanto as métricas. Os erros de Água classificado como Fogo costumam ser Pokémon de Água **vermelhos ou laranjas** (caranguejos, peixes vermelhos), o que confirma que o modelo se apoia muito na cor. Já os de Fogo classificado como Água costumam ser Pokémon de fogo com cores frias ou claras. Esse é o limite de usar só cor e forma simples."

> ⚠️ Os erros abaixo usam o limiar ajustado para F2, que é baixo e gera muitos alarmes falsos (Água classificado como Fogo). Isso é o trade-off do F2 aparecendo nas imagens.

In [ ]:
def mostrar_erros(nome, tipo_erro, n=8):
    # tipo_erro: 'fp' = Água classificado como Fogo (alarme falso) | 'fn' = Fogo classificado como Água (Fogo perdido)
    p = oof[(PRINCIPAL, nome)]
    pred = (p >= thr_otimo[nome]).astype(int)
    if tipo_erro == 'fp':
        idx = np.where((y == 0) & (pred == 1))[0]
        ordem = idx[np.argsort(-p[idx])]          # mais confiantes no erro primeiro (maior P(Fogo))
        titulo = 'Água classificado como Fogo'
    else:
        idx = np.where((y == 1) & (pred == 0))[0]
        ordem = idx[np.argsort(p[idx])]           # mais confiantes no erro primeiro (menor P(Fogo))
        titulo = 'Fogo classificado como Água'
    print(f'{nome}: {len(idx)} erros do tipo "{titulo}"')
    ordem = ordem[:n]
    if len(ordem) == 0:
        return
    fig, axes = plt.subplots(1, len(ordem), figsize=(2.2 * len(ordem), 3.0), squeeze=False)
    for ax, i in zip(axes[0], ordem):
        ax.imshow(Image.open(df['path'].iloc[i]).convert('RGBA'))
        ax.axis('off')
        ax.set_title(f"{df['name'].iloc[i]}\nP(Fogo)={p[i]:.2f}", fontsize=8)
    fig.suptitle(f'{nome}: {titulo}')
    plt.tight_layout(); plt.show()

for nome in ['SVM', 'Random Forest']:
    mostrar_erros(nome, 'fp')   # Água classificado como Fogo
    mostrar_erros(nome, 'fn')   # Fogo classificado como Água

## 5. Predição com probabilidade

Saída no formato pedido no enunciado: classe e probabilidade (ex.: Fogo, 0.81).

> 🎤 **O que falar:** "O requisito era devolver a classe junto com a probabilidade, no estilo 'dog, 0,81'. Aqui mostro 6 Pokémon aleatórios. A probabilidade vem de um modelo que **não viu aquele Pokémon** no treino (previsão da validação cruzada). Um valor perto de 0,5 indica um caso duvidoso."

In [ ]:
def classe_e_prob(p_fogo, limiar):
    # Devolve a classe prevista e a probabilidade DESSA classe (se previu Água, é 1 - p_fogo)
    if p_fogo >= limiar:
        return 'Fogo', round(float(p_fogo), 2)
    return 'Água', round(float(1 - p_fogo), 2)

amostra = df.sample(6, random_state=SEED)
for i, linha in amostra.iterrows():
    svm = classe_e_prob(oof[(PRINCIPAL, 'SVM')][i], thr_otimo['SVM'])
    rf = classe_e_prob(oof[(PRINCIPAL, 'Random Forest')][i], thr_otimo['Random Forest'])
    print(f'{linha["name"]:<14} real: {linha.classe:<5} | SVM: {svm} | RF: {rf}')

## 6. Melhorias: ajuste de parâmetros e mais modelos

Duas melhorias sobre o conjunto de features principal:
1. **Ajuste de hiperparâmetros** (`GridSearchCV`): cada modelo testa uma pequena grade de parâmetros.
2. **Mais modelos:** KNN e Gradient Boosting, além de SVM e Random Forest.

**Validação cruzada aninhada:** o ajuste de parâmetros roda **dentro** de cada fold de treino. Assim, a imagem que está sendo avaliada nunca influencia a escolha dos parâmetros, e o resultado não fica inflado.

> 🎤 **O que falar:** "Fiz duas melhorias. Primeiro, ajustei os parâmetros de cada modelo com busca em grade. Segundo, incluí KNN e Gradient Boosting. Para não me enganar, a busca de parâmetros roda dentro de cada fold da validação cruzada (validação aninhada), então os parâmetros nunca são escolhidos olhando para a imagem que está sendo avaliada."

> ⚠️ O KNN não tem `class_weight`, então ignora o desbalanceamento; o ajuste do limiar compensa parte disso. A célula de validação aninhada leva alguns minutos.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.base import clone

def criar_modelos_tuning():
    # Cada modelo vem com uma grade pequena de parâmetros; a busca usa validação cruzada interna (3 folds)
    interna = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    def busca(modelo, grade):
        return GridSearchCV(modelo, grade, cv=interna, scoring='roc_auc', n_jobs=-1)   # critério: ROC AUC
    return {
        'SVM': busca(make_pipeline(StandardScaler(),
                                   SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=SEED)),
                     {'svc__C': [0.5, 1, 4, 16], 'svc__gamma': ['scale', 0.003, 0.01]}),
        'Random Forest': busca(RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=1, random_state=SEED),
                               {'max_depth': [None, 6, 12], 'min_samples_leaf': [1, 2, 4]}),
        'KNN': busca(make_pipeline(StandardScaler(), KNeighborsClassifier()),
                     {'kneighborsclassifier__n_neighbors': [3, 5, 9, 15],
                      'kneighborsclassifier__weights': ['uniform', 'distance']}),
        'Gradient Boosting': busca(HistGradientBoostingClassifier(class_weight='balanced', random_state=SEED),
                                   {'learning_rate': [0.03, 0.1], 'max_depth': [2, 3, None], 'max_iter': [100, 200]}),
    }

X_p = X[PRINCIPAL]    # features do conjunto principal

In [ ]:
# Validação cruzada ANINHADA: o GridSearchCV (busca de parâmetros) roda dentro de cada fold de treino
oof_t = {}
for nome, m in criar_modelos_tuning().items():
    print('Validação cruzada aninhada:', nome, '...')
    oof_t[nome] = cross_val_predict(m, X_p, y, cv=cv, method='predict_proba')[:, 1]
print('Pronto.')

### 6.1 Comparação dos modelos ajustados (limiar 0,5 e limiar ajustado para F2)

> 🎤 **O que falar:** "Esta tabela compara os quatro modelos ajustados. Olho principalmente o ROC AUC, que não depende do limiar, e depois o F2 com o limiar ajustado."

In [ ]:
thr_t, linhas = {}, []
for nome, p in oof_t.items():
    f2s = [fbeta_score(y, (p >= t).astype(int), beta=2, zero_division=0) for t in limiares]
    thr_t[nome] = float(limiares[int(np.argmax(f2s))])        # limiar que maximiza o F2 (mesma regra de antes)
    linhas.append({'Modelo': nome, 'Limiar': 0.50, **metricas(y, p, 0.50)})
    linhas.append({'Modelo': nome, 'Limiar': round(thr_t[nome], 2), **metricas(y, p, thr_t[nome])})

comparacao = pd.DataFrame(linhas).round(3)
comparacao

In [ ]:
# Antes (modelos com parâmetros padrão) vs depois (ajustados): ROC AUC na validação cruzada
auc_base = {f'{n} (base)': roc_auc_score(y, oof[(PRINCIPAL, n)]) for n in ['SVM', 'Random Forest']}
auc_novo = {f'{n} (ajustado)': roc_auc_score(y, p) for n, p in oof_t.items()}
tabela_auc = pd.Series({**auc_base, **auc_novo}).sort_values()
tabela_auc.plot(kind='barh', figsize=(7, 4), title='ROC AUC (validação cruzada)',
                xlim=(max(0.5, tabela_auc.min() - 0.05), 1.0))
plt.xlabel('ROC AUC'); plt.show()

# Matriz de confusão dos dois melhores modelos por ROC AUC (limiar ajustado para F2)
individuais = ['SVM', 'Random Forest', 'KNN', 'Gradient Boosting']
ranking = sorted(individuais, key=lambda n: -roc_auc_score(y, oof_t[n]))
melhor = ranking[0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, nome in zip(axes, ranking[:2]):
    pred = (oof_t[nome] >= thr_t[nome]).astype(int)
    ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=['Água', 'Fogo']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome} (limiar={thr_t[nome]:.2f})')
plt.tight_layout(); plt.show()

### 6.2 Modelos finais

Agora cada modelo busca os melhores parâmetros em **todos** os dados e é treinado uma última vez. Esses modelos são os que vão para o app.

In [ ]:
finais, params = {}, {}
for nome, m in criar_modelos_tuning().items():
    m.fit(X_p, y)                          # busca os melhores parâmetros e treina com todas as imagens
    finais[nome] = m.best_estimator_
    params[nome] = m.best_params_
    print(nome, '->', m.best_params_)

## 7. Teste de robustez: variantes shiny e em tons de cinza

Um modelo baseado em cor deveria piorar quando a cor muda. Vamos testar isso com o dataset *Pokémon Dataset 1000* (Kaggle), que traz cerca de 26 imagens por espécie: sprites de jogos antigos, arte oficial, versões **shiny** (de outra cor) e versões em **tons de cinza**.

**Como o teste funciona (sem vazamento):** usamos as mesmas divisões da validação cruzada, por espécie. Em cada fold, treinamos com a imagem **original** das espécies de treino e testamos nas **variantes** das espécies de teste. O modelo nunca viu a espécie que está sendo testada.

> 🎤 **O que falar:** "Quero saber o quanto o modelo depende da cor. Peguei um dataset com várias versões de cada Pokémon, incluindo versões shiny, que têm cor diferente da normal, e versões em tons de cinza. Treino com a imagem original e testo nessas variantes, sempre em espécies que o modelo não viu. Se o desempenho cair, é um indício de que o modelo se apoia na cor."

> ⚠️ **Limitações:** o rótulo de cada variante é o tipo da espécie (vem do nosso CSV, pois esse dataset não tem tipos). Não conseguimos marcar quais imagens são shiny; só separamos automaticamente as em tons de cinza. Como o dataset mistura estilos (pixel art e arte oficial), a queda de desempenho pode ter outras causas além da cor.

In [ ]:
p2 = kagglehub.dataset_download('noodulz/pokemon-dataset-1000')
arqs = glob.glob(os.path.join(p2, '**', 'dataset', '*', '*.png'), recursive=True)   # usa a pasta 'dataset' (todas as imagens)
if len(arqs) == 0:
    raise ValueError('Não encontrei imagens em .../dataset/<espécie>/*.png. Confira a estrutura da pasta.')

variantes = {}      # espécie (nome normalizado) -> lista de imagens
for f in arqs:
    especie = norm(os.path.basename(os.path.dirname(f)))
    variantes.setdefault(especie, []).append(f)
print('Espécies no dataset grande:', len(variantes), '| imagens:', len(arqs))

df['sp'] = df['name'].map(norm)
tem_var = df['sp'].isin(variantes).values
print('Espécies do nosso conjunto com variantes:', int(tem_var.sum()), 'de', len(df))

In [ ]:
from features import _mascara
from skimage.color import rgb2hsv

K = 6   # no máximo 6 variantes por espécie (mantém o teste rápido)
rng = np.random.RandomState(SEED)
linhas_v = []
for i, linha in df.iterrows():
    if linha['sp'] in variantes:
        lista = sorted(variantes[linha['sp']])
        for f in rng.choice(lista, size=min(K, len(lista)), replace=False):
            linhas_v.append({'idx': i, 'path': f, 'label': linha['label']})   # idx = posição da espécie em df
dv = pd.DataFrame(linhas_v)

def eh_cinza(f):
    # Imagem "sem cor": saturação média dos pixels do Pokémon muito baixa
    a = np.asarray(Image.open(f).convert('RGBA').resize((SIZE, SIZE)), dtype=np.float32) / 255.0
    mask = _mascara(a[..., :3], a[..., 3])
    return bool(rgb2hsv(a[..., :3])[..., 1][mask].mean() < 0.08)

dv['cinza'] = dv['path'].map(eh_cinza)
X_var = np.array([extrair(f, PRINCIPAL) for f in dv['path']])      # mesmas features do treino
print(len(dv), 'variantes |', int(dv['cinza'].sum()), 'em tons de cinza')

In [ ]:
# Em cada fold: treina com a imagem ORIGINAL das espécies de treino e testa nas VARIANTES das espécies de teste
prob_var = {n: np.zeros(len(dv)) for n in finais}
for tr, te in cv.split(X_p, y):                      # mesmas divisões (por espécie) da validação cruzada
    na_te = dv['idx'].isin(te).values
    for nome, m in finais.items():
        modelo = clone(m).fit(X_p[tr], y[tr])        # clone mantém os melhores parâmetros, sem o treino anterior
        prob_var[nome][na_te] = modelo.predict_proba(X_var[na_te])[:, 1]

def metricas_seguro(y_real, proba, limiar):
    # Igual a metricas(), mas não quebra se o grupo tiver uma classe só (o ROC AUC precisa das duas)
    try:
        return metricas(y_real, proba, limiar)
    except ValueError:
        return {k: np.nan for k in ['Acurácia', 'Precisão', 'Recall', 'F1', 'F2', 'ROC AUC']}

y_v, cinza = dv['label'].values, dv['cinza'].values
idx_c = np.where(tem_var)[0]                         # espécies que têm variantes (comparação justa com o original)
linhas = []
for nome in prob_var:
    cenarios = [
        ('Original (mesmas espécies)', y[idx_c], oof_t[nome][idx_c]),
        ('Variantes (todas)', y_v, prob_var[nome]),
        ('Variantes coloridas', y_v[~cinza], prob_var[nome][~cinza]),
        ('Variantes em cinza', y_v[cinza], prob_var[nome][cinza]),
    ]
    for cenario, yy, pp in cenarios:
        if len(yy) > 0:
            linhas.append({'Modelo': nome, 'Teste': cenario, 'n': len(yy), **metricas_seguro(yy, pp, thr_t[nome])})
robustez = pd.DataFrame(linhas).round(3)
robustez

### 7.1 Original vs variantes

> 🎤 **Como ler:** compare, para cada modelo, o F2 em "Original" com o F2 em "Variantes coloridas" e "Variantes em cinza". Se cair nas variantes coloridas, o modelo sofre com cores diferentes das normais (shiny) e com estilos diferentes de imagem. Se cair muito nas em cinza, mostra que ele depende da cor: sem cor, sobra só a forma.

In [ ]:
ordem_cols = [c for c in ['Original (mesmas espécies)', 'Variantes coloridas', 'Variantes em cinza'] if c in robustez['Teste'].values]
robustez.pivot(index='Modelo', columns='Teste', values='F2')[ordem_cols].plot(kind='bar', figsize=(9, 4), title='F2: original vs variantes')
plt.ylabel('F2'); plt.xticks(rotation=20); plt.show()

### 7.1.1 É o limiar ou é a imagem?

Nas variantes, a acurácia cai bem mais que o ROC AUC. Uma explicação possível: o limiar do F2 foi ajustado nas imagens originais e **não se ajusta** às variantes (os modelos passam a dizer "Fogo" demais). Para testar, comparamos nas variantes três limiares:
1. **O mesmo do F2** (o que usamos até aqui).
2. **O padrão, 0,50.**
3. **O melhor limiar das variantes, pelo F1** (um "oráculo": usa os rótulos das próprias variantes, então é **otimista** e serve só de diagnóstico, não de resultado). Usamos o F1 e não o F2 porque o F2 favorece o recall e voltaria a escolher limiares baixos.

Também mostramos a **% prevista como Fogo** e a acurácia do chute "sempre Água".

> 🎤 **O que falar:** "Se, com um limiar melhor, a acurácia nas variantes volta para perto do original, o problema era o limiar, não a imagem. Se continuar baixa mesmo com o melhor limiar possível, as variantes são realmente mais difíceis. O limiar 'oráculo' é trapaça de propósito: mostra o melhor caso, só para separar as duas explicações."

> ⚠️ O limiar "oráculo" usa os rótulos das variantes. Não use esses números como desempenho do modelo.

In [ ]:
base_orig = 1 - y[idx_c].mean()      # acurácia do chute 'sempre Água' nas imagens originais
base_var = 1 - y_v.mean()            # idem nas variantes
print(f'Chute "sempre Água": {base_orig:.3f} (originais) | {base_var:.3f} (variantes)')

def linha_lim(nome, cenario, yy, pp, thr):
    pred = (pp >= thr).astype(int)
    m = metricas_seguro(yy, pp, thr)
    return {'Modelo': nome, 'Cenário': cenario, 'Limiar': round(float(thr), 2),
            'Acurácia': m['Acurácia'], 'Precisão': m['Precisão'], 'Recall': m['Recall'], 'F2': m['F2'],
            '% previsto Fogo': 100 * pred.mean()}

linhas = []
for nome in prob_var:
    pv = prob_var[nome]
    f1s = [f1_score(y_v, (pv >= t).astype(int), zero_division=0) for t in limiares]
    thr_oraculo = float(limiares[int(np.argmax(f1s))])                       # melhor limiar (F1) nas variantes: oráculo
    linhas += [
        linha_lim(nome, 'Original (limiar F2)', y[idx_c], oof_t[nome][idx_c], thr_t[nome]),
        linha_lim(nome, 'Variantes (limiar F2)', y_v, pv, thr_t[nome]),
        linha_lim(nome, 'Variantes (limiar 0,50)', y_v, pv, 0.50),
        linha_lim(nome, 'Variantes (limiar oráculo)', y_v, pv, thr_oraculo),
    ]
robustez_limiar = pd.DataFrame(linhas).round(3)
robustez_limiar

In [ ]:
# Acurácia por cenário, com a linha do chute "sempre Água"
ac = robustez_limiar.pivot(index='Modelo', columns='Cenário', values='Acurácia')[
    ['Original (limiar F2)', 'Variantes (limiar F2)', 'Variantes (limiar 0,50)', 'Variantes (limiar oráculo)']]
ax = ac.plot(kind='bar', figsize=(10, 4.5), title='Acurácia: o limiar explica a queda nas variantes?')
ax.axhline(base_var, color='gray', linestyle='--', label='chute "sempre Água"')
ax.set_ylabel('Acurácia'); ax.legend(fontsize=8); plt.xticks(rotation=20); plt.show()

### 7.2 Que variantes enganam o SVM?

> 🎤 **O que falar:** "Aqui estão as variantes que o SVM errou com mais confiança. Espero ver versões shiny, com cor diferente da normal, e versões em cinza. Esse é um bom exemplo de que o modelo aprendeu a cor, e não o tipo."

In [ ]:
def mostrar_erros_var(nome, tipo_erro, n=8):
    # tipo_erro: 'fp' = Água classificado como Fogo | 'fn' = Fogo classificado como Água
    p = prob_var[nome]
    pred = (p >= thr_t[nome]).astype(int)
    if tipo_erro == 'fp':
        idx = np.where((y_v == 0) & (pred == 1))[0]
        ordem = idx[np.argsort(-p[idx])]
        titulo = 'Água classificado como Fogo'
    else:
        idx = np.where((y_v == 1) & (pred == 0))[0]
        ordem = idx[np.argsort(p[idx])]
        titulo = 'Fogo classificado como Água'
    print(f'{nome}: {len(idx)} erros do tipo "{titulo}"')
    ordem = ordem[:n]
    if len(ordem) == 0:
        return
    fig, axes = plt.subplots(1, len(ordem), figsize=(2.2 * len(ordem), 3.0), squeeze=False)
    for ax, i in zip(axes[0], ordem):
        ax.imshow(Image.open(dv['path'].iloc[i]).convert('RGBA'))
        ax.axis('off')
        ax.set_title(df['name'].iloc[dv['idx'].iloc[i]], fontsize=8)
        ax.text(0.5, -0.08, f'P(Fogo)={p[i]:.2f}', transform=ax.transAxes, ha='center', fontsize=8)
    fig.suptitle(f'{nome}: {titulo}')
    plt.tight_layout(); plt.show()

mostrar_erros_var('SVM', 'fp')
mostrar_erros_var('SVM', 'fn')

## 8. Extensão: 3 classes (Fogo, Água, Outro)

Até aqui o modelo só conhece Fogo e Água, então um Pokémon de outro tipo (ex.: Graveler, de Pedra) é **forçado** a cair em uma das duas. Nesta extensão acrescentamos a classe **Outro**: qualquer Pokémon que não seja de Fogo nem de Água (usamos também os Pokémon dos outros tipos, que o conjunto binário descartava).

> ⚠️ O enunciado pede classificação **binária**. Esta seção é uma **extensão separada**; o resultado principal do trabalho continua sendo o das seções anteriores.

**Cuidados**
- "Outro" reúne os outros 16 tipos e é a classe **majoritária**, então a acurácia sozinha engana. Usamos **acurácia balanceada**, **F1 macro** e o recall de cada classe.
- Aqui não usamos o limiar do F2: a decisão é pela **maior probabilidade** (argmax).
- Usamos os mesmos parâmetros dos modelos já ajustados (seção 6), sem nova busca.

> 🎤 **O que falar:** "Percebi que o modelo binário erra quando aparece um Pokémon que não é de Fogo nem de Água, como o Graveler, porque ele é obrigado a escolher uma das duas. Então testei uma extensão com uma terceira classe, 'Outro'. É um problema bem mais difícil: 'Outro' junta muitos tipos diferentes e é a classe maior. Por isso olho o recall de cada classe e a acurácia balanceada, e não só a acurácia."

In [ ]:
# Todos os Pokémon, menos os que são Fogo E Água ao mesmo tempo (ambíguos)
df3 = meta[~(meta.fogo & meta.agua)].copy().reset_index(drop=True)
NOMES3 = ['Água', 'Fogo', 'Outro']
df3['y3'] = np.where(df3.fogo, 1, np.where(df3.agua, 0, 2))      # 0 = Água, 1 = Fogo, 2 = Outro
y3 = df3['y3'].values

print(pd.Series(y3).map(dict(enumerate(NOMES3))).value_counts())
print(f'Chute "sempre Outro": acurácia {np.mean(y3 == 2):.3f}')

X3 = np.array([extrair(p, PRINCIPAL) for p in df3.path])         # mesmas features dos modelos finais
print('Features:', X3.shape)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support

# Validação cruzada estratificada (mesmo esquema de antes). Cada modelo devolve 3 probabilidades por imagem
oof3 = {}
for nome, m in finais.items():
    print('Validação cruzada (3 classes):', nome, '...')
    oof3[nome] = cross_val_predict(clone(m), X3, y3, cv=cv, method='predict_proba')     # formato (n, 3)

def metricas3(y_real, proba):
    pred = proba.argmax(axis=1)                      # classe com maior probabilidade
    prec, rec, f1, _ = precision_recall_fscore_support(y_real, pred, labels=[0, 1, 2], zero_division=0)
    return {'Acurácia': accuracy_score(y_real, pred), 'Acurácia balanceada': balanced_accuracy_score(y_real, pred),
            'F1 macro': f1.mean(),
            'Recall Água': rec[0], 'Recall Fogo': rec[1], 'Recall Outro': rec[2],
            'Precisão Água': prec[0], 'Precisão Fogo': prec[1], 'Precisão Outro': prec[2]}

resumo3 = pd.DataFrame([{'Modelo': n, **metricas3(y3, p)} for n, p in oof3.items()]).round(3)
resumo3

### 8.1 Matrizes de confusão (3 classes)

> 🎤 **Como ler:** linhas = classe real, colunas = classe prevista. A diagonal são os acertos. Olhe a linha "Outro": quantos Pokémon de outros tipos o modelo manda para Fogo ou Água? E a coluna "Outro": quantos Fogo e Água ele joga para "Outro"?

In [ ]:
ranking3 = list(resumo3.sort_values('F1 macro', ascending=False)['Modelo'])      # modelos do melhor ao pior F1 macro
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, nome in zip(axes, ranking3[:2]):
    cm = confusion_matrix(y3, oof3[nome].argmax(axis=1), labels=[0, 1, 2])
    ConfusionMatrixDisplay(cm, display_labels=NOMES3).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nome)
plt.tight_layout(); plt.show()

### 8.2 Ajuste da decisão: um peso para "Outro"

Como "Outro" é a classe maior, a decisão pela maior probabilidade a favorece e o recall de Fogo e Água fica baixo. A ideia é a mesma do limiar do F2 no binário: **multiplicar a probabilidade de "Outro" por um peso entre 0,05 e 1,00** antes de escolher a classe. Peso 1,00 = sem ajuste; peso menor = o modelo diz "Outro" com menos facilidade.

Escolhemos, para cada modelo, o peso que maximiza o **F1 macro** (que dá o mesmo valor às 3 classes).

> 🎤 **O que falar:** "Como 'Outro' é a classe maior, o modelo tende a escolhê-la. Para compensar, multipliquei a probabilidade de 'Outro' por um peso menor que 1 e escolhi o peso que maximiza o F1 macro. É o mesmo raciocínio do limiar do F2 no problema binário: troca-se um pouco de precisão por mais recall nas classes pequenas."

> ⚠️ O peso foi escolhido nas mesmas previsões usadas para avaliar, então o resultado é um pouco otimista.

In [ ]:
pesos = np.round(np.arange(0.05, 1.001, 0.05), 2)       # candidatos: 0,05 a 1,00

def prever_com_peso(proba, w):
    # Multiplica só a coluna 'Outro' (índice 2) pelo peso e escolhe a classe de maior valor
    return (proba * np.array([1.0, 1.0, w])).argmax(axis=1)

def metricas3_pred(y_real, pred):
    prec, rec, f1, _ = precision_recall_fscore_support(y_real, pred, labels=[0, 1, 2], zero_division=0)
    return {'Acurácia': accuracy_score(y_real, pred), 'Acurácia balanceada': balanced_accuracy_score(y_real, pred),
            'F1 macro': f1.mean(),
            'Recall Água': rec[0], 'Recall Fogo': rec[1], 'Recall Outro': rec[2],
            'Precisão Água': prec[0], 'Precisão Fogo': prec[1], 'Precisão Outro': prec[2]}

w_otimo, curvas, linhas = {}, {}, []
for nome, proba in oof3.items():
    curvas[nome] = [f1_score(y3, prever_com_peso(proba, w), average='macro') for w in pesos]
    w_otimo[nome] = float(pesos[int(np.argmax(curvas[nome]))])           # peso com maior F1 macro
    linhas.append({'Modelo': nome, 'Peso de Outro': 1.0, **metricas3_pred(y3, prever_com_peso(proba, 1.0))})
    linhas.append({'Modelo': nome, 'Peso de Outro': w_otimo[nome], **metricas3_pred(y3, prever_com_peso(proba, w_otimo[nome]))})
resumo3_peso = pd.DataFrame(linhas).round(3)
resumo3_peso

In [ ]:
# F1 macro em função do peso de 'Outro': onde está o melhor ponto de cada modelo?
plt.figure(figsize=(8, 4.5))
for nome, curva in curvas.items():
    plt.plot(pesos, curva, label=f'{nome} (melhor peso = {w_otimo[nome]:.2f})')
plt.xlabel('Peso aplicado a P(Outro)  (1,00 = sem ajuste)'); plt.ylabel('F1 macro')
plt.title('Efeito do peso de "Outro" no F1 macro'); plt.legend(fontsize=8); plt.show()

# Matriz de confusão do melhor modelo (maior F1 macro com ajuste), sem ajuste e com ajuste
melhor_w = max(curvas, key=lambda n: max(curvas[n]))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (titulo, w) in zip(axes, [('sem ajuste (peso 1,00)', 1.0), (f'com ajuste (peso {w_otimo[melhor_w]:.2f})', w_otimo[melhor_w])]):
    cm = confusion_matrix(y3, prever_com_peso(oof3[melhor_w], w), labels=[0, 1, 2])
    ConfusionMatrixDisplay(cm, display_labels=NOMES3).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(melhor_w + ' ' + titulo)
plt.tight_layout(); plt.show()

### 8.3 Exemplos

> 🎤 **O que falar:** "Aqui estão alguns Pokémon conhecidos, com a probabilidade que o melhor modelo deu para cada classe, e depois os Pokémon de outros tipos que o modelo confunde com Fogo ou Água."

In [ ]:
exemplos = ['graveler', 'kingler', 'vaporeon', 'charizard', 'magikarp', 'bulbasaur', 'pikachu']
sel = df3[df3['name'].map(norm).isin(exemplos)]
for i, linha in sel.iterrows():
    p = oof3[melhor_w][i]
    com_ajuste = NOMES3[int(prever_com_peso(p.reshape(1, -1), w_otimo[melhor_w])[0])]     # decisão com o peso de Outro
    print(f'{linha["name"]:<12} real: {NOMES3[y3[i]]:<5} | ' + melhor_w + ': ' + ' | '.join(f'{c} {v:.2f}' for c, v in zip(NOMES3, p))
          + f' | com ajuste: {com_ajuste}')

def mostrar_outro_errado(nome, para, n=8):
    # Pokémon de 'Outro' que o modelo mandou para Fogo (para=1) ou Água (para=0), os mais confiantes primeiro
    pred = oof3[nome].argmax(axis=1)
    idx = np.where((y3 == 2) & (pred == para))[0]
    ordem = idx[np.argsort(-oof3[nome][idx, para])][:n]
    print(f'{nome}: {len(idx)} Pokémon de "Outro" classificados como {NOMES3[para]}')
    if len(ordem) == 0:
        return
    fig, axes = plt.subplots(1, len(ordem), figsize=(2.2 * len(ordem), 3.0), squeeze=False)
    for ax, i in zip(axes[0], ordem):
        ax.imshow(Image.open(df3['path'].iloc[i]).convert('RGBA')); ax.axis('off')
        ax.set_title(f"{df3['name'].iloc[i]}\nP({NOMES3[para]})={oof3[nome][i, para]:.2f}", fontsize=8)
    fig.suptitle(f'Outro classificado como {NOMES3[para]} ({nome})')
    plt.tight_layout(); plt.show()

mostrar_outro_errado(melhor_w, 1)    # Outro -> Fogo
mostrar_outro_errado(melhor_w, 0)    # Outro -> Água

In [ ]:
# Modelos finais de 3 classes: treinados com todos os Pokémon desta seção (vão para o app)
finais3 = {nome: clone(m).fit(X3, y3) for nome, m in finais.items()}
print('Modelos de 3 classes treinados:', list(finais3))

## 9. Exportar para o app Streamlit

Salva os modelos finais (já ajustados e treinados com todas as imagens), os limiares, as tabelas de resultados e o `features.py` em **um arquivo zip**.

> 🎤 **O que falar:** "Para o app não precisar retreinar, salvo os modelos em arquivos. A avaliação foi feita por validação cruzada; o modelo final usa todos os dados. O app importa o mesmo `features.py`, então o pré-processamento é idêntico ao do treino."

In [ ]:
import joblib, sklearn, zipfile
from google.colab import files

# Um arquivo por modelo (SVM, Random Forest, KNN, Gradient Boosting)
arquivos_modelos = {}
for nome, m in finais.items():
    arq = 'modelo_' + re.sub(r'[^a-z0-9]', '', nome.lower()) + '.joblib'
    joblib.dump(m, arq, compress=3)
    arquivos_modelos[nome] = arq

# Configuração lida pelo app: conjunto de features, arquivos dos modelos e limiares (F2)
# Extensão de 3 classes (seção 8): um arquivo por modelo
arquivos3 = {}
for nome, m in finais3.items():
    arq = 'modelo3_' + re.sub(r'[^a-z0-9]', '', nome.lower()) + '.joblib'
    joblib.dump(m, arq, compress=3)
    arquivos3[nome] = arq

json.dump({'conjunto': PRINCIPAL, 'modelos': arquivos_modelos, 'limiares': thr_t,
           'tres_classes': {'conjunto': PRINCIPAL, 'modelos': arquivos3, 'classes': NOMES3, 'pesos_outro': w_otimo}},
          open('config.json', 'w'))

# Tabelas para as abas de métricas do app
comparacao.to_csv('resultados.csv', index=False)
ablacao.to_csv('ablacao.csv', index=False)
robustez.to_csv('robustez.csv', index=False)
robustez_limiar.to_csv('robustez_limiar.csv', index=False)
resumo3.to_csv('tres_classes.csv', index=False)
resumo3_peso.to_csv('tres_classes_peso.csv', index=False)

# Nomes por número da Pokédex: o app mostra o nome quando o arquivo enviado tem o número (ex.: 099.png) ou o nome (kingler.png).
# Hipótese: no CSV do dataset as linhas estão na ordem da Pokédex nacional (linha 1 = nº 001). A conferência abaixo confirma.
nomes_csv = pd.read_csv(csv_path)
nomes_csv.columns = [c.strip().lower() for c in nomes_csv.columns]
pokedex = {str(i + 1): str(n) for i, n in enumerate(nomes_csv['name'])}
print('Conferência (esperado: kingler, graveler, vaporeon):', pokedex.get('99'), pokedex.get('75'), pokedex.get('134'))
json.dump(pokedex, open('pokedex.json', 'w', encoding='utf-8'), ensure_ascii=False)

# Junta tudo em um zip (um download só)
lista = (list(arquivos_modelos.values()) + list(arquivos3.values()) +
         ['config.json', 'resultados.csv', 'ablacao.csv', 'robustez.csv', 'robustez_limiar.csv', 'tres_classes.csv', 'tres_classes_peso.csv', 'pokedex.json', 'features.py'])
with zipfile.ZipFile('arquivos_app.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for arq in lista:
        z.write(arq)

# IMPORTANTE: o app deve usar a MESMA versão do scikit-learn do Colab, senão o joblib pode falhar
print('scikit-learn do Colab:', sklearn.__version__)
files.download('arquivos_app.zip')

## 10. Pontos para o relatório e a apresentação
- **Hipótese:** cor importa mais que forma para o tipo. A ablação (cor, HOG, cor+HOG, cor extra) responde isso com dados.
- **Conjunto pequeno:** por isso validação cruzada estratificada em vez de um teste minúsculo.
- **Desbalanceamento:** `class_weight='balanced'` e a acurácia sozinha enganando.
- **Limiar e F2:** efeito de mover o limiar de 0,5 para o ótimo (recall sobe, precisão cai).
- **Melhorias:** ajuste de hiperparâmetros com validação aninhada, mais dois modelos (KNN e Gradient Boosting). Compare "base" e "ajustado" no gráfico de ROC AUC.
- **Robustez:** treino na imagem original e teste em variantes shiny e em cinza, por espécie. Mostra o quanto o modelo depende da cor.
- **Limitações:** evoluções parecidas entre si podem deixar o resultado otimista; um Pokémon pode ter cores que não combinam com o tipo (ex.: um Pokémon de Água vermelho); o rótulo das variantes vem da espécie, e não conseguimos identificar quais são shiny; no problema binário não existe a classe "Outro", então o modelo é forçado a escolher entre Fogo e Água (a extensão da seção 8 testa 3 classes).

- **Extensão de 3 classes:** a classe "Outro" resolve casos como o Graveler, mas derruba o recall de Fogo e Água; o peso de "Outro" recupera parte disso, ao custo de mais alarmes falsos. Resultados no README.

> 🎤 **Possíveis perguntas da banca e respostas curtas:**
> - *Por que não usou rede neural?* O enunciado pedia modelos clássicos (KNN, RF, GB, SVM), então extraí features manualmente.
> - *Por que validação cruzada e não treino/teste?* O conjunto é pequeno; um teste de 15% teria poucas dezenas de imagens e as métricas oscilariam muito.
> - *Por que F2?* Pesa o recall 2x mais; mostra o efeito de priorizar não perder nenhum Fogo. Aqui é ilustrativo, já que o problema é lúdico.
> - *A cor realmente importa?* Sim. Na ablação, a cor sozinha deu ROC AUC de 0,895 (SVM) e 0,911 (Random Forest), e o HOG sozinho ficou perto do acaso (0,627 e 0,689). Juntar o HOG piorou o SVM (0,833). Nas variantes shiny e em cinza o AUC cai de ~0,90 para ~0,85.
> - *Os parâmetros foram escolhidos olhando o teste?* Não: a busca roda dentro de cada fold (validação aninhada).
> - *Onde o modelo erra?* Em Pokémon cuja cor não combina com o tipo, e em variantes shiny; dá para mostrar exemplos.
> - *Por que só Fogo e Água?* Para ficar binário; escolhi tipos com cores bem associadas, o que ajuda a interpretar o resultado.
> - *E um Pokémon de outro tipo?* No modelo binário ele é forçado a cair em uma das duas. Testei uma extensão com a classe "Outro"; é mais difícil porque "Outro" é grande e heterogênea. A acurácia (0,74 a 0,80) fica perto do chute de "sempre Outro" (~0,76), então olho a acurácia balanceada (0,48 a 0,67) e o F1 macro (0,52 a 0,61). Um peso para "Outro" sobe o F1 macro do SVM para 0,63, mas o Charizard e o Growlithe continuam errados, porque usamos só cor.